<a href="https://colab.research.google.com/github/Jlnlys/machine-learning-from-scratch/blob/main/LogisticRegression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Logistic Regression**

This is the formula for the logistic function: $f(x) = \frac{1}{1 + e^{Ax + b}}$.

$Logloss = \frac{1}{n}\sum_{i=1}^{n}y_i*log(\hat{y_i}) + (1-y_i)* log(1-\hat{y_i})$

To calculte thge gradiant we need to derivate on W and on B

so we will have:

$\frac{\partial E}{\partial w} = \frac{1}{n}\sum x_i (-y_i + \hat{y_i})$

$\frac{\partial E}{\partial b} = \frac{1}{n}\sum (-y_i +  \hat{y_i})$


In [ ]:
import numpy as np

In [ ]:
class My_LogisticRegression():
  def __init__(self):
    self.w = None
    self.b = None

  def my_sigmoid(self,X):
    return 1/(1+np.exp(-(np.dot(X,self.w) + self.b)))

  def gradient(self,X,y_true,y_pred):
    sub = np.subtract(y_pred,y_true)
    dw =  np.dot(X.T, sub) / len(sub)
    db = np.mean(sub)
    return dw,db

  def fit(self,X,y,lr=0.01,epochs=10000):
    if len(X.shape) == 1:
      X = X.reshape(-1,1)
    self.w = np.zeros(X.shape[1])
    self.b = 0
    for i in range(epochs):
      y_pred = self.my_sigmoid(X)
      dw,db = self.gradient(X,y,y_pred)
      self.w = self.w - lr*dw
      self.b = self.b - lr*db

  def predict(self,X):
    y_pred = self.my_sigmoid(X)
    y_pred = np.where(y_pred>0.5,1,0)
    return y_pred


In [ ]:
from sklearn.linear_model import LogisticRegression
import pandas as pd

In [ ]:
cols = ["fLength","fWidth","fSize","fConc","fConc1","fAsym","fM3Long","fM3Trans","fAlpha","fDist","class"]
df = pd.read_csv("magic04.data",names=cols)
df.head()

,fLength,fWidth,fSize,fConc,fConc1,fAsym,fM3Long,fM3Trans,fAlpha,fDist,class
0,28.7967,16.0021,2.6449,0.3918,0.1982,27.7004,22.0110,-8.2027,40.0920,81.8828,g
1,31.6036,11.7235,2.5185,0.5303,0.3773,26.2722,23.8238,-9.9574,6.3609,205.2610,g
2,162.0520,136.0310,4.0612,0.0374,0.0187,116.7410,-64.8580,-45.2160,76.9600,256.7880,g
3,23.8172,9.5728,2.3385,0.6147,0.3922,27.2107,-6.4633,-7.1513,10.4490,116.7370,g
4,75.1362,30.9205,3.1611,0.3168,0.1832,-5.5277,28.5525,21.8393,4.6480,356.4620,g


In [ ]:
df["class"] = (df["class"] == "g").astype(int)
df.head()

,fLength,fWidth,fSize,fConc,fConc1,fAsym,fM3Long,fM3Trans,fAlpha,fDist,class
0,28.7967,16.0021,2.6449,0.3918,0.1982,27.7004,22.0110,-8.2027,40.0920,81.8828,1
1,31.6036,11.7235,2.5185,0.5303,0.3773,26.2722,23.8238,-9.9574,6.3609,205.2610,1
2,162.0520,136.0310,4.0612,0.0374,0.0187,116.7410,-64.8580,-45.2160,76.9600,256.7880,1
3,23.8172,9.5728,2.3385,0.6147,0.3922,27.2107,-6.4633,-7.1513,10.4490,116.7370,1
4,75.1362,30.9205,3.1611,0.3168,0.1832,-5.5277,28.5525,21.8393,4.6480,356.4620,1


In [ ]:
train, valid, test = np.split(df.sample(frac=1),[int(0.6*len(df)),int(0.8*len(df))])

/usr/local/lib/python3.12/dist-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


In [ ]:
print(len(train),len(valid),len(test))
print(len(train[train["class"] == 1]))
print(len(train[train["class"] == 0]))

11412 3804 3804
7377
4035


In [ ]:
from imblearn.over_sampling import RandomOverSampler
from sklearn.preprocessing import StandardScaler

In [ ]:
def standardize(dataframe,random=True):
  x = dataframe[dataframe.columns[:-1]].values
  y = dataframe[dataframe.columns[-1]].values
  scaler = StandardScaler()
  x = scaler.fit_transform(x)
  if random:
    x,y = RandomOverSampler().fit_resample(x,y)

  data = np.hstack((x,np.reshape(y,(-1,1))))

  return data,x,y

In [ ]:
train_data, train_x, train_y = standardize(train,True)
valid_data, valid_x, valid_y = standardize(valid,False)
test_data, test_x, test_y = standardize(test,False)

In [ ]:
from sklearn.metrics import  classification_report

In [ ]:
My_lr = My_LogisticRegression()
My_lr.fit(train_x,train_y)
y_pred = My_lr.predict(test_x)
print(classification_report(test_y,y_pred))


              precision    recall  f1-score   support

           0       0.68      0.72      0.70      1332
           1       0.84      0.82      0.83      2472

    accuracy                           0.78      3804
   macro avg       0.76      0.77      0.77      3804
weighted avg       0.79      0.78      0.79      3804



In [ ]:
sk_lr = LogisticRegression()
sk_lr.fit(train_x,train_y)
y_pred = sk_lr.predict(test_x)
print(classification_report(test_y,y_pred))


              precision    recall  f1-score   support

           0       0.68      0.72      0.70      1332
           1       0.84      0.82      0.83      2472

    accuracy                           0.78      3804
   macro avg       0.76      0.77      0.77      3804
weighted avg       0.79      0.78      0.79      3804

